In [1]:
import pandas as pd
import pandas_ta_classic as ta
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau


import numpy as np

import random
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, precision_score, recall_score, f1_score, matthews_corrcoef


In [2]:
master_df = pd.read_parquet("stocknet-dataset/master_df.parquet")
master_df.reset_index(drop=True, inplace=True)

print("shape before sector features:", master_df.shape)
print("Columns in master_df:", master_df.columns)

shape before sector features: (104220, 39)
Columns in master_df: Index(['date', 'open', 'high', 'low', 'close', 'adj close', 'volume', 'ticker',
       'text', 'sentiment', 'emotion_anger', 'emotion_disgust', 'emotion_fear',
       'emotion_joy', 'emotion_neutral', 'emotion_sadness', 'emotion_surprize',
       'stance_positive', 'stance_negative', 'sector', 'company_name',
       'EMA_12', 'EMA_26', 'EMA_50', 'MACD_12_26_9', 'MACDh_12_26_9',
       'MACDs_12_26_9', 'RSI_14', 'STOCHRSIk_14_14_3_3', 'STOCHRSId_14_14_3_3',
       'ATRr_14', 'BB_upper', 'BB_middle', 'BB_lower', 'OBV', 'ret_1d',
       'roll_ret_1d', 'roll_ret_5d', 'roll_ret_20d'],
      dtype='object')


In [3]:
master_df['date'] = pd.to_datetime(master_df['date'])
master_df = master_df.sort_values(by=['ticker', 'date']).reset_index(drop=True)

In [4]:
H_META = 1 # 1-day ahead meta target

In [5]:
master_df = master_df.sort_values(by=['ticker', 'date'])

master_df['ret_1d_meta'] = (
    master_df.groupby('ticker')['close']
    .pct_change(periods=-H_META)
)

In [6]:
master_df['up_1d_meta'] = (master_df['ret_1d_meta'] > 0).astype(int)

master_df['has_meta_target'] = master_df['ret_1d_meta'].notna().astype(int)

In [7]:
base_feature_columns = [
    'open', 'high', 'low', 'close', 'volume',
    'roll_ret_1d', 'roll_ret_5d', 
    'roll_ret_20d',
    
    'stance_positive', 'stance_negative', 'sentiment',
                   
    # 'sector_open_mean', 'sector_high_mean', 'sector_low_mean', 'sector_volume_mean',
    
    'EMA_12', 'EMA_26', 'EMA_50', 'MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9',
    'RSI_14', 'ATRr_14', 'STOCHRSIk_14_14_3_3', 'STOCHRSId_14_14_3_3',
    'BB_upper', 'BB_middle', 'BB_lower', 'OBV',
    
    # 'sector_ret_1d', 'sector_ret_5d', 'sector_ret_20d', 
    # # 'sector_ret_60d',
    # 'sector_vol_20d', 'sector_dispersion_1d',
    # 'sector_rel_strength',
    
    # 'EMA_12_sector', 'EMA_26_sector', 'EMA_50_sector', 'MACD_12_26_9_sector', 'MACDh_12_26_9_sector', 
    # 'MACDs_12_26_9_sector', 'RSI_14_sector', 'sector_BB_upper', 'sector_BB_middle','sector_BB_lower'
]

In [8]:
tickers = master_df['ticker'].unique()
len(tickers), tickers[:5]


(88, array(['AAPL', 'ABB', 'ABBV', 'AEP', 'AGFS'], dtype=object))

In [9]:
def split_ticker_meta(df_ticker, cutoff_date):
    df_ticker = df_ticker.sort_values('date')
    # remove rows without target
    df_ticker = df_ticker[~df_ticker['ret_1d_meta'].isna()].copy()
    if df_ticker.empty:
        return None, None
    
    train_mask = df_ticker['date'] <= cutoff_date
    test_mask  = df_ticker['date'] > cutoff_date
    
    train_df = df_ticker[train_mask]
    test_df  = df_ticker[test_mask]
    
    # need enough data to train
    if len(train_df) < 200 or len(test_df) < 20:
        return None, None
    
    return train_df, test_df


In [10]:
def set_global_seeds(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_global_seeds(42)


# Datasets
class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class RNNHead(nn.Module):
    # Shared head:
    #   - RNN stack (LSTM/GRU, uni/bi)
    #   - BatchNorm + Dense(32, ReLU) + Dropout
    #   - Output layer (1 unit): linear (regression) or logits (classification)
    def __init__(self, input_size, rnn_type='LSTM', bidirectional=False, problem_type='classification',
                 n_classes=6, ordinal_head='coral', hidden1=128, hidden2=64, num_layers=1,
                 inter_rnn_drop=0.1, dropout=0.3, use_layernorm=False):
        super().__init__()
        self.problem_type = problem_type
        self.bidirectional = bidirectional
        self.rnn_type = rnn_type.upper()
        self.n_classes = n_classes
        self.ordinal_head = ordinal_head
        self.num_layers = int(num_layers)
        self.hidden1 = int(hidden1)
        self.hidden2 = int(hidden2)

        if self.num_layers not in (1, 2):
            raise ValueError("num_layers must be 1 or 2")

        rnn_cls = {'LSTM': nn.LSTM, 'GRU': nn.GRU}[('GRU' if 'GRU' in self.rnn_type else 'LSTM')]

        self.rnn1 = rnn_cls(
            input_size=input_size, hidden_size=self.hidden1, num_layers=1,
            batch_first=True, dropout=0.0, bidirectional=bidirectional
        )

        self.inter_rnn_drop = nn.Dropout(float(inter_rnn_drop))

        self.rnn2 = None
        if self.num_layers == 2:
            self.rnn2 = rnn_cls(
                input_size=self.hidden1*(2 if bidirectional else 1), hidden_size=self.hidden2, num_layers=1,
                batch_first=True, dropout=0.0, bidirectional=bidirectional
            )
            feat_dim = self.hidden2*(2 if bidirectional else 1)
        else:
            feat_dim = self.hidden1*(2 if bidirectional else 1)

        if use_layernorm:
            self.bn = nn.LayerNorm(feat_dim)
        else:
            self.bn = nn.BatchNorm1d(feat_dim)

        self.fc = nn.Linear(feat_dim, 32)
        self.drop = nn.Dropout(float(dropout))
        self.out = nn.Linear(32, 1)

    def forward(self, x):
        # x: [B, T, F]
        out, _ = self.rnn1(x)
        if self.num_layers == 2:
            out = self.inter_rnn_drop(out)   # inter-layer dropout (sequence-wise)
            out, _ = self.rnn2(out)
        # take last timestep: [B, T, H] -> [B, H]
        out = out[:, -1, :]
        out = self.bn(out)
        out = F.relu(self.fc(out))
        out = self.drop(out)
        out = self.out(out)  # shape [B,1]
        return out  # regression: raw; classification: logits
# Early Stopping (PyTorch)
class EarlyStopper:
    def __init__(self, patience=15, min_delta=0.0, restore_best=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best = restore_best
        self.best_loss = float('inf')
        self.counter = 0
        self.best_state = None

    def step(self, val_loss, model):
        improved = (self.best_loss - val_loss) > self.min_delta
        if improved:
            self.best_loss = val_loss
            self.counter = 0
            if self.restore_best:
                # Deep copy state dict
                self.best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.restore_best and self.best_state is not None:
            model.load_state_dict(self.best_state)


In [11]:
def build_model(input_shape, model_type='LSTM', problem_type='regression', hidden1=128, hidden2=64, 
                num_layers=2, inter_rnn_drop=0.1, dropout=0.3):
    seq_len, n_features = input_shape
    model_type = model_type.upper()
    kwargs = dict(
        problem_type=problem_type,
        hidden1=hidden1,
        hidden2=hidden2,
        num_layers=num_layers,
        inter_rnn_drop=inter_rnn_drop,
        dropout=dropout,
    )
    if model_type == 'LSTM':
        return RNNHead(n_features, rnn_type='LSTM', bidirectional=False, **kwargs)
    elif model_type == 'BILSTM':
        return RNNHead(n_features, rnn_type='LSTM', bidirectional=True, **kwargs)
    elif model_type == 'GRU':
        return RNNHead(n_features, rnn_type='GRU', bidirectional=False, **kwargs)
    elif model_type == 'BIGRU':
        return RNNHead(n_features, rnn_type='GRU', bidirectional=True, **kwargs)
    else:
        raise ValueError("Model type must be one of: ['LSTM','BiLSTM','GRU','BiGRU']")

In [ ]:
L_REGRESSION = 18     # sequence length for meta regression
REFIT_INTERVAL = 20   # refit cadence in trading days
W_BASE = 300          # warm-up period before first predictions
MIN_SEQ = 50          # minimum sequences required to train


def ffill_only(df, cols):
    return df.assign(**{c: df[c].ffill() for c in cols})

def fit_scaler_on_train(train_df, feature_cols):
    scaler = StandardScaler()
    scaler.fit(train_df[feature_cols].values)
    return scaler

def transform_df(df, scaler, feature_cols):
    out = df.copy()
    out[feature_cols] = scaler.transform(df[feature_cols].values)
    return out

def build_sequences_meta(df, feature_cols, target_col, seq_len):
    df = df.sort_values('date').reset_index(drop=True)
    X_vals = df[feature_cols].values
    y_vals = df[target_col].values
    X_list, y_list, idx_list = [], [], []
    for i in range(seq_len - 1, len(df)):
        window_X = X_vals[i-seq_len+1:i+1]  # window ending at time t
        y = y_vals[i]                       # forward return t -> t+1
        if np.isnan(window_X).any() or pd.isna(y):
            continue
        X_list.append(window_X)
        y_list.append(y)
        idx_list.append(i)
    return np.array(X_list), np.array(y_list), idx_list


def walk_forward_regression_predictions_per_ticker(
    df_tkr,
    feature_cols,
    target_col='ret_1d_meta',
    seq_len=L_REGRESSION,
    w_base=W_BASE,
    refit_interval=REFIT_INTERVAL,
    device=None
):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # log model and params used
    print(f"[META-REG] model=LSTM seq_len={seq_len} w_base={w_base} refit={refit_interval} hidden1=64 hidden2=32 num_layers=2 inter_drop=0.4 dropout=0.0 lr=1e-5 wd=4e-4")

    df_tkr = df_tkr.sort_values('date').reset_index(drop=True)
    df_tkr = df_tkr[~df_tkr[target_col].isna()].copy().reset_index(drop=True)
    if len(df_tkr) < (w_base + refit_interval + seq_len):
        return None, "too_short"

    df_tkr = ffill_only(df_tkr, feature_cols)
    mask_all = df_tkr[feature_cols].notna().all(axis=1)
    if not mask_all.any():
        return None, "no_valid_rows"
    first_good_idx = mask_all.idxmax()
    df_tkr = df_tkr.iloc[first_good_idx:].reset_index(drop=True)
    if len(df_tkr) < (w_base + refit_interval + seq_len):
        return None, "too_short_after_ffill"

    rows = []
    metrics_rows = []
    k = max(w_base - 1, seq_len - 1)

    while True:
        train_end = k
        pred_start = k + 1
        pred_end = min(k + refit_interval, len(df_tkr) - 1)
        if pred_start > pred_end:
            break

        train_df = df_tkr.iloc[:train_end + 1].copy()
        train_df[feature_cols] = train_df[feature_cols].ffill()

        scaler = fit_scaler_on_train(train_df, feature_cols)
        train_df_s = transform_df(train_df, scaler, feature_cols)

        seq_X_train, seq_y_train, _ = build_sequences_meta(train_df_s, feature_cols, target_col, seq_len)
        if len(seq_X_train) < MIN_SEQ:
            return None, "too_few_sequences"

        val_cut = max(int(len(seq_X_train) * 0.2), 1)
        n_train = len(seq_X_train) - val_cut
        X_tr, y_tr = seq_X_train[:n_train], seq_y_train[:n_train]
        X_val, y_val = seq_X_train[n_train:], seq_y_train[n_train:]
        if len(X_val) == 0:
            X_val, y_val = X_tr, y_tr

        train_ds = SequenceDataset(X_tr, y_tr)
        val_ds   = SequenceDataset(X_val, y_val)
        train_loader = DataLoader(train_ds, batch_size=32, shuffle=False, drop_last=False)
        val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, drop_last=False)

        model = build_model((seq_len, X_tr.shape[-1]), model_type='LSTM', problem_type='regression', num_layers=2, hidden1=64, hidden2=32, inter_rnn_drop=0.4, dropout=0.0).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-5, weight_decay=4e-4)
        loss_fn = nn.HuberLoss(delta=1.0)
        early = EarlyStopper(patience=15, min_delta=0.0, restore_best=True)

        for epoch in range(50):
            model.train()
            for xb, yb in train_loader:
                xb = xb.to(device)
                yb = yb.to(device).view(-1, 1)
                optimizer.zero_grad(set_to_none=True)
                preds = model(xb)
                loss = loss_fn(preds, yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            model.eval()
            with torch.no_grad():
                vtot, n = 0.0, 0
                for xb, yb in val_loader:
                    xb = xb.to(device)
                    yb = yb.to(device).view(-1, 1)
                    preds = model(xb)
                    loss = loss_fn(preds, yb)
                    vtot += loss.item() * xb.size(0)
                    n += xb.size(0)
                val_loss = vtot / max(n, 1)

            if early.step(val_loss, model):
                break

        early.restore(model)

        block_end = pred_end
        block_df = df_tkr.iloc[:block_end + 1].copy()
        block_df[feature_cols] = block_df[feature_cols].ffill()
        block_df_s = transform_df(block_df, scaler, feature_cols)

        seq_X_all, seq_y_all, idx_all = build_sequences_meta(block_df_s, feature_cols, target_col, seq_len)
        if len(seq_X_all) == 0:
            del model
            if device.type == 'cuda':
                torch.cuda.empty_cache()
            break

        end_dates = block_df_s.iloc[idx_all]['date'].values
        block_dates = df_tkr.iloc[pred_start:pred_end+1]['date'].values
        block_mask = np.isin(end_dates, block_dates)

        if block_mask.any():
            X_block = torch.tensor(seq_X_all[block_mask], dtype=torch.float32).to(device)
            y_block = seq_y_all[block_mask]
            model.eval()
            with torch.no_grad():
                preds = model(X_block).view(-1).cpu().numpy()

            mae = mean_absolute_error(y_block, preds)
            mse = mean_squared_error(y_block, preds)
            y_true_dir = (y_block > 0).astype(int)
            y_pred_dir = (preds > 0).astype(int)
            precision = precision_score(y_true_dir, y_pred_dir, zero_division=0)
            recall = recall_score(y_true_dir, y_pred_dir, zero_division=0)
            f1 = f1_score(y_true_dir, y_pred_dir, zero_division=0)
            acc = (y_true_dir == y_pred_dir).mean()
            mcc = matthews_corrcoef(y_true_dir, y_pred_dir)

            metrics_rows.append({
                'ticker': df_tkr.loc[0, 'ticker'],
                'refit_date': df_tkr.iloc[train_end]['date'],
                'block_end': df_tkr.iloc[pred_end]['date'],
                'mae': mae,
                'mse': mse,
                'dir_acc': acc,
                'f1': f1,
                'precision': precision,
                'recall': recall,
                'mcc': mcc,
                'n_preds': int(block_mask.sum())
            })

            for d, p in zip(end_dates[block_mask], preds):
                rows.append({'date': d, 'ticker': df_tkr.loc[0, 'ticker'], 'regression_pred_ret_1d': float(p)})

        del model
        if device.type == 'cuda':
            torch.cuda.empty_cache()

        k += refit_interval
        if k >= len(df_tkr) - 2:
            break

    pred_df = pd.DataFrame(rows)
    metrics_df = pd.DataFrame(metrics_rows)
    return pred_df, metrics_df


print(f"Using Regression meta sequence length: {L_REGRESSION}, refit interval: {REFIT_INTERVAL}, warm-up: {W_BASE}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

all_preds = []
all_metrics = []
skip_counts = {}

for tkr in tickers:
    print(f"Processing {tkr} for walk-forward Regression predictions...")
    df_tkr = master_df[master_df['ticker'] == tkr].copy()
    pred_df, metrics_df = walk_forward_regression_predictions_per_ticker(
        df_tkr,
        feature_cols=base_feature_columns,
        target_col='ret_1d_meta',
        seq_len=L_REGRESSION,
        w_base=W_BASE,
        refit_interval=REFIT_INTERVAL,
        device=device
    )
    if pred_df is None or pred_df.empty:
        reason = metrics_df if isinstance(metrics_df, str) else 'no_predictions'
        skip_counts[reason] = skip_counts.get(reason, 0) + 1
        print(f"  Skipping {tkr}: {reason}")
        continue

    all_preds.append(pred_df)
    if metrics_df is not None and not metrics_df.empty:
        all_metrics.append(metrics_df)
    print(f"  Stored {len(pred_df)} predictions for {tkr}")

regression_meta_df = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame()
metrics_df_all = pd.concat(all_metrics, ignore_index=True) if all_metrics else pd.DataFrame()
print("Regression OOS prediction rows:", regression_meta_df.shape)
print("Skip summary:", skip_counts)

if not metrics_df_all.empty:
    agg = metrics_df_all[['mae','mse','dir_acc','f1','precision','recall','mcc']].mean().to_dict()
    total_preds = metrics_df_all['n_preds'].sum()
    print("Aggregate Regression meta metrics (across refits/tickers):")
    print(f"  MAE: {agg['mae']:.6f}, MSE: {agg['mse']:.6f}")
    print(f"  Dir Acc: {agg['dir_acc']:.4f}, F1: {agg['f1']:.4f}, Precision: {agg['precision']:.4f}, Recall: {agg['recall']:.4f}, MCC: {agg['mcc']:.4f}")
    print(f"  Refits: {len(metrics_df_all)}, Total preds: {total_preds}")
else:
    print("No metrics to aggregate (no refits processed)")


Using GRU meta sequence length: 18, refit interval: 20, warm-up: 300
Processing AAPL for walk-forward GRU predictions...
[META-REG] model=LSTM seq_len=18 w_base=300 refit=20 hidden1=64 hidden2=32 num_layers=2 inter_drop=0.4 dropout=0.0 lr=1e-5 wd=4e-4


KeyboardInterrupt: 

In [ ]:
# drop stale prediction columns before merging
# ensure predictions dataframe exists
try:
    regression_meta_df
except NameError:
    regression_meta_df = pd.DataFrame(columns=['date','ticker','regression_pred_ret_1d'])

if 'regression_pred_ret_1d' in master_df.columns:
    master_df = master_df.drop(columns=['regression_pred_ret_1d'])

# merge new regression predictions
master_df = master_df.merge(regression_meta_df, on=['date', 'ticker'], how='left')
print('master_df with regression meta:', master_df.shape)

# compute regression prediction errors (diagnostics)
mask = master_df['regression_pred_ret_1d'].notna() & master_df['ret_1d_meta'].notna()
master_df.loc[mask, 'regression_err_ret_1d']     = master_df.loc[mask, 'regression_pred_ret_1d'] - master_df.loc[mask, 'ret_1d_meta']
master_df.loc[mask, 'regression_abs_err_ret_1d'] = master_df.loc[mask, 'regression_err_ret_1d'].abs()
master_df.loc[mask, 'regression_sq_err_ret_1d']  = master_df.loc[mask, 'regression_err_ret_1d'] ** 2
master_df.loc[mask, 'regression_dir_correct_1d'] = (
    (master_df.loc[mask, 'regression_pred_ret_1d'] > 0).astype(int) == master_df.loc[mask, 'up_1d_meta']
).astype(int)

# leak-safe reliability features (shifted/rolling)
master_df = master_df.sort_values(['ticker', 'date']).reset_index(drop=True)
master_df['regression_abs_err_lag1'] = master_df.groupby('ticker')['regression_abs_err_ret_1d'].shift(1)
master_df['regression_mae_20'] = (
    master_df.groupby('ticker')['regression_abs_err_ret_1d']
    .transform(lambda s: s.shift(1).rolling(20, min_periods=5).mean())
)
master_df['regression_rmse_20'] = (
    master_df.groupby('ticker')['regression_sq_err_ret_1d']
    .transform(lambda s: np.sqrt(s.shift(1).rolling(20, min_periods=5).mean()))
)
master_df['regression_dir_acc_20'] = (
    master_df.groupby('ticker')['regression_dir_correct_1d']
    .transform(lambda s: s.shift(1).rolling(20, min_periods=5).mean())
)

print("Reliability features computed. Sample:")
print(master_df[['date','ticker','regression_pred_ret_1d','regression_mae_20','regression_dir_acc_20']].head())

In [ ]:

# Walk-forward classification (probability of up move)
L_CLASSIFICATION = 12
REFIT_INTERVAL_CLASSIFICATION = 20
W_BASE_CLASSIFICATION = 300
MIN_SEQ_CLASSIFICATION = 50


def walk_forward_classification_predictions_per_ticker(
    df_tkr,
    feature_cols
    ,
    target_col='up_1d_meta',
    seq_len=L_CLASSIFICATION,
    w_base=W_BASE_CLASSIFICATION,
    refit_interval=REFIT_INTERVAL_CLASSIFICATION,
    device=None
):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # log model and params used
    print(f"[META-CLS] model=GRU seq_len={seq_len} w_base={w_base} refit={refit_interval} hidden1=64 hidden2=32 num_layers=2 inter_drop=0.0 dropout=0.4 lr=1e-5 wd=4e-4")

    df_tkr = df_tkr.sort_values('date').reset_index(drop=True)
    df_tkr = df_tkr[~df_tkr[target_col].isna()].copy().reset_index(drop=True)
    if len(df_tkr) < (w_base + refit_interval + seq_len):
        return None, "too_short"

    df_tkr = ffill_only(df_tkr, feature_cols)
    mask_all = df_tkr[feature_cols].notna().all(axis=1)
    if not mask_all.any():
        return None, "no_valid_rows"
    first_good_idx = mask_all.idxmax()
    df_tkr = df_tkr.iloc[first_good_idx:].reset_index(drop=True)
    if len(df_tkr) < (w_base + refit_interval + seq_len):
        return None, "too_short_after_ffill"

    rows = []
    metrics_rows = []
    k = max(w_base - 1, seq_len - 1)

    while True:
        train_end = k
        pred_start = k + 1
        pred_end = min(k + refit_interval, len(df_tkr) - 1)
        if pred_start > pred_end:
            break

        train_df = df_tkr.iloc[:train_end + 1].copy()
        train_df[feature_cols] = train_df[feature_cols].ffill()

        scaler = fit_scaler_on_train(train_df, feature_cols)
        train_df_s = transform_df(train_df, scaler, feature_cols)

        seq_X_train, seq_y_train, _ = build_sequences_meta(train_df_s, feature_cols, target_col, seq_len)
        if len(seq_X_train) < MIN_SEQ_CLASSIFICATION:
            return None, "too_few_sequences"

        val_cut = max(int(len(seq_X_train) * 0.2), 1)
        n_train = len(seq_X_train) - val_cut
        X_tr, y_tr = seq_X_train[:n_train], seq_y_train[:n_train]
        X_val, y_val = seq_X_train[n_train:], seq_y_train[n_train:]
        if len(X_val) == 0:
            X_val, y_val = X_tr, y_tr

        pos_rate = y_tr.mean()
        if pos_rate > 0 and pos_rate < 1:
            pos_weight = (1.0 - pos_rate) / pos_rate
            pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32, device=device)
        else:
            pos_weight_tensor = None

        train_ds = SequenceDataset(X_tr, y_tr)
        val_ds   = SequenceDataset(X_val, y_val)
        train_loader = DataLoader(train_ds, batch_size=32, shuffle=False, drop_last=False)
        val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, drop_last=False)

        model = build_model((seq_len, X_tr.shape[-1]), model_type='GRU', problem_type='classification', num_layers=1, hidden1=256, hidden2=64, inter_rnn_drop=0.0, dropout=0.4).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=7e-3, weight_decay=2e-3)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
        early = EarlyStopper(patience=5, min_delta=0, restore_best=True)

        for epoch in range(20):
            model.train()
            for xb, yb in train_loader:
                xb = xb.to(device)
                yb = yb.to(device).view(-1, 1)
                optimizer.zero_grad(set_to_none=True)
                logits = model(xb)
                loss = loss_fn(logits, yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            model.eval()
            with torch.no_grad():
                vtot, n = 0.0, 0
                for xb, yb in val_loader:
                    xb = xb.to(device)
                    yb = yb.to(device).view(-1, 1)
                    logits = model(xb)
                    loss = loss_fn(logits, yb)
                    vtot += loss.item() * xb.size(0)
                    n += xb.size(0)
                val_loss = vtot / max(n, 1)

            if early.step(val_loss, model):
                break

        early.restore(model)

        block_end = pred_end
        block_df = df_tkr.iloc[:block_end + 1].copy()
        block_df[feature_cols] = block_df[feature_cols].ffill()
        block_df_s = transform_df(block_df, scaler, feature_cols)

        seq_X_all, seq_y_all, idx_all = build_sequences_meta(block_df_s, feature_cols, target_col, seq_len)
        if len(seq_X_all) == 0:
            del model
            if device.type == 'cuda':
                torch.cuda.empty_cache()
            break

        end_dates = block_df_s.iloc[idx_all]['date'].values
        block_dates = df_tkr.iloc[pred_start:pred_end+1]['date'].values
        block_mask = np.isin(end_dates, block_dates)

        if block_mask.any():
            X_block = torch.tensor(seq_X_all[block_mask], dtype=torch.float32).to(device)
            y_block = seq_y_all[block_mask]
            model.eval()
            with torch.no_grad():
                logits_block = model(X_block).view(-1).cpu().numpy()
            probs_block = 1.0 / (1.0 + np.exp(-logits_block))

            eps = 1e-8
            brier = (probs_block - y_block) ** 2
            logloss = - (y_block * np.log(probs_block + eps) + (1 - y_block) * np.log(1 - probs_block + eps))
            correct = (probs_block >= 0.5).astype(int) == y_block

            metrics_rows.append({
                'ticker': df_tkr.loc[0, 'ticker'],
                'refit_date': df_tkr.iloc[train_end]['date'],
                'block_end': df_tkr.iloc[pred_end]['date'],
                'brier': float(brier.mean()),
                'logloss': float(logloss.mean()),
                'acc': float(correct.mean()),
                'n_preds': int(block_mask.sum())
            })

            for d, p, lgt in zip(end_dates[block_mask], probs_block, logits_block):
                rows.append({'date': d, 'ticker': df_tkr.loc[0, 'ticker'], 'prob_up_1d': float(p), 'logit_up_1d': float(lgt)})

        del model
        if device.type == 'cuda':
            torch.cuda.empty_cache()

        k += refit_interval
        if k >= len(df_tkr) - 2:
            break

    pred_df = pd.DataFrame(rows)
    metrics_df = pd.DataFrame(metrics_rows)
    return pred_df, metrics_df


print(f"Using classification: seq {L_CLASSIFICATION}, refit interval {REFIT_INTERVAL_CLASSIFICATION}, warm-up {W_BASE_CLASSIFICATION}")
all_classification_preds = []
all_classification_metrics = []
skip_counts_classification = {}

base_feature_columns.extend(['roll_ret_20d'])

for tkr in tickers:
    print(f"Processing {tkr} for walk-forward Classification...")
    df_tkr = master_df[master_df['ticker'] == tkr].copy()
    pred_df, metrics_df = walk_forward_classification_predictions_per_ticker(
        df_tkr,
        feature_cols=base_feature_columns,
        target_col='up_1d_meta',
        seq_len=L_CLASSIFICATION,
        w_base=W_BASE_CLASSIFICATION,
        refit_interval=REFIT_INTERVAL_CLASSIFICATION,
        device=device
    )
    if pred_df is None or pred_df.empty:
        reason = metrics_df if isinstance(metrics_df, str) else 'no_predictions'
        skip_counts_classification[reason] = skip_counts_classification.get(reason, 0) + 1
        print(f"  Skipping {tkr}: {reason}")
        continue

    all_classification_preds.append(pred_df)
    if metrics_df is not None and not metrics_df.empty:
        all_classification_metrics.append(metrics_df)
    print(f"  Stored {len(pred_df)} classification predictions for {tkr}")

classification_pred_df = pd.concat(all_classification_preds, ignore_index=True) if all_classification_preds else pd.DataFrame()
classification_metrics_all = pd.concat(all_classification_metrics, ignore_index=True) if all_classification_metrics else pd.DataFrame()
print("Classification OOS prediction rows:", classification_pred_df.shape)
print("Classification skip summary:", skip_counts_classification)

if not classification_metrics_all.empty:
    agg = classification_metrics_all[['brier','logloss','acc']].mean().to_dict()
    total_preds = classification_metrics_all['n_preds'].sum()
    print("Aggregate Classification metrics (across refits/tickers):")
    print(f"  Brier: {agg['brier']:.6f}, Logloss: {agg['logloss']:.6f}, Acc: {agg['acc']:.4f}")
    print(f"  Refits: {len(classification_metrics_all)}, Total preds: {total_preds}")
else:
    print("No Classification metrics to aggregate (no refits processed)")


In [ ]:
# drop stale Classification prediction columns before merging
if 'prob_up_1d' in master_df.columns:
    master_df = master_df.drop(columns=['prob_up_1d'])
if 'logit_up_1d' in master_df.columns:
    master_df = master_df.drop(columns=['logit_up_1d'])

# ensure classification_pred_df exists
try:
    classification_pred_df
except NameError:
    classification_pred_df = pd.DataFrame(columns=['date','ticker','prob_up_1d','logit_up_1d'])

master_df = master_df.merge(classification_pred_df, on=['date','ticker'], how='left')
print('master_df with Classification meta:', master_df.shape)

# diagnostics and leak-safe reliability features for Classification
mask_cls = master_df['prob_up_1d'].notna() & master_df['up_1d_meta'].notna()
eps = 1e-8
master_df.loc[mask_cls, 'brier'] = (master_df.loc[mask_cls, 'prob_up_1d'] - master_df.loc[mask_cls, 'up_1d_meta']) ** 2
master_df.loc[mask_cls, 'logloss'] = -(
    master_df.loc[mask_cls, 'up_1d_meta'] * np.log(master_df.loc[mask_cls, 'prob_up_1d'] + eps)
    + (1 - master_df.loc[mask_cls, 'up_1d_meta']) * np.log(1 - master_df.loc[mask_cls, 'prob_up_1d'] + eps)
)
master_df.loc[mask_cls, 'correct'] = (
    (master_df.loc[mask_cls, 'prob_up_1d'] >= 0.5).astype(int) == master_df.loc[mask_cls, 'up_1d_meta']
).astype(int)

master_df = master_df.sort_values(['ticker','date']).reset_index(drop=True)
master_df['brier_20'] = (
    master_df.groupby('ticker')['brier']
    .transform(lambda s: s.shift(1).rolling(20, min_periods=5).mean())
)
master_df['logloss_20'] = (
    master_df.groupby('ticker')['logloss']
    .transform(lambda s: s.shift(1).rolling(20, min_periods=5).mean())
)
master_df['acc_20'] = (
    master_df.groupby('ticker')['correct']
    .transform(lambda s: s.shift(1).rolling(20, min_periods=5).mean())
)

print("Classification reliability features computed. Sample:")
print(master_df[['date','ticker','prob_up_1d','brier_20','acc_20']].head())

# save combined parquet with Regression and Classification signals
out_path = "stocknet-dataset/master_df_meta_base.parquet"
master_df.to_parquet(out_path, index=False)
print("Saved:", out_path)


In [ ]:
random.seed(42)
sector_ticker_map = {}
for sector in master_df['sector'].dropna().unique():
    tickers_in_sector = master_df.loc[master_df['sector'] == sector, 'ticker'].dropna().unique()
    if len(tickers_in_sector):
        sector_ticker_map[sector] = random.choice(tickers_in_sector)

for sector, ticker in sector_ticker_map.items():
    stock_data = master_df[master_df['ticker'] == ticker].dropna(subset=['regression_pred_ret_1d', 'ret_1d_meta'])
    if stock_data.empty:
        print(f"No valid data for sector {sector}, ticker {ticker}. Skipping.")
        continue
    stock_data = stock_data.sort_values('date')
    plt.figure(figsize=(10, 6))
    plt.plot(stock_data['date'], stock_data['ret_1d_meta'], label='Actual Return', color='blue')
    plt.plot(stock_data['date'], stock_data['regression_pred_ret_1d'], label='Regression Pred Return', color='orange')
    plt.fill_between(
        stock_data['date'],
        stock_data['regression_pred_ret_1d'] - stock_data['regression_abs_err_ret_1d'],
        stock_data['regression_pred_ret_1d'] + stock_data['regression_abs_err_ret_1d'],
        color='orange', alpha=0.2, label='|Error| Band'
    )
    plt.title(f"{sector} - {ticker}: Actual vs Predicted Return")
    plt.xlabel("Date")
    plt.ylabel("1-day return")
    plt.legend()
    plt.grid()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

# Load the dataset
file_path = "stocknet-dataset/master_df_meta_base.parquet"
df = pd.read_parquet(file_path)

# Display basic information about the dataset
print("Dataset Info:")
print(df.info())
print("\nDataset Description:")
print(df.describe(include='all'))

# Check for missing values
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({'Feature': missing_values.index, 'Missing Values': missing_values.values, 'Percentage': missing_percentage.values})
print("\nMissing Values:")
print(missing_df.sort_values(by='Percentage', ascending=False))

# Visualize missing values
plt.figure(figsize=(10, 6))
sns.barplot(x='Percentage', y='Feature', data=missing_df.sort_values(by='Percentage', ascending=False).head(20))
plt.title("Top 20 Features with Missing Values")
plt.xlabel("Percentage of Missing Values")
plt.ylabel("Feature")
plt.show()

# Correlation analysis
numerical_features = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numerical_features].corr()

# Plot heatmap of correlations
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, cmap='coolwarm', annot=False, fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

# Identify highly correlated features
threshold = 0.8
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        if abs(correlation_matrix.iloc[i, j]) > threshold:
            high_corr_pairs.append((correlation_matrix.columns[i], correlation_matrix.columns[j], correlation_matrix.iloc[i, j]))

print("\nHighly Correlated Features (Threshold > 0.8):")
for pair in high_corr_pairs:
    print(f"{pair[0]} and {pair[1]}: {pair[2]:.2f}")

# Visualize distributions of numerical features
for feature in numerical_features:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[feature].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {feature}")
    plt.xlabel(feature)
    plt.ylabel("Frequency")
    plt.show()